# 실습 ① — 내 손으로 만드는 고장 예측 모델
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abh-edu/mfg-ai-course/blob/main/lab1_failure_prediction.ipynb)

**제조 AI 심화과정 · 2~3교시** — 설비 가동 기록 10,000건으로 고장 예측 모델을 만들고, 그 성적표를 의심해 봅니다.

- 칸 왼쪽의 ▶ 를 **위에서 아래로 차례로** 누릅니다. 코드는 읽지 않으셔도 됩니다.
- 빨간 글씨(Warning)는 대부분 정상입니다. **Error** 가 보이면 그 칸을 한 번 더 실행해 보세요.
- 꼬이면 [런타임] → [세션 다시 시작]을 누르고 첫 칸부터 다시 실행합니다. GPU는 필요 없습니다.

**2교시는 STEP 0부터 4까지, 3교시는 STEP 5부터 7까지**입니다.

In [ ]:
# [STEP 0] 준비 — 도구를 올리고, 데이터를 데려옵니다 (약 20초)
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# 그래프에 한글이 보이도록 글꼴 준비 (실패해도 실습에는 지장이 없습니다)
try:
    os.system("pip install -q koreanize-matplotlib"); import koreanize_matplotlib
except Exception:
    print("한글 글꼴을 준비하지 못했습니다. 그래프의 글자가 깨져 보일 수 있습니다.")
plt.rcParams["axes.unicode_minus"] = False

URLS = [
    "https://raw.githubusercontent.com/abh-edu/mfg-ai-course/main/ai4i2020.csv",        # 수업 공식 저장소
    "https://raw.githubusercontent.com/Naveen1402/ai4i2020/main/ai4i2020.csv",          # 예비 경로
]
df = None
for u in URLS:
    try:
        df = pd.read_csv(u)
        break
    except Exception:
        continue
assert df is not None, "데이터를 불러오지 못했습니다 — 이 칸을 한 번 더 실행해 보세요."
assert df.shape == (10000, 14), f"예상과 다른 데이터입니다: {df.shape}"
print(f"준비 완료 {df.shape}")

`준비 완료 (10000, 14)` 가 보이면 성공입니다 — **설비 가동 기록 10,000건, 기록마다 값 14개.**

## STEP 1 — 데이터 첫 대면
열(컬럼) 14개의 뜻:

| 열 | 뜻 | 비고 |
|---|---|---|
| UDI / Product ID | 일련번호 / 제품 ID | 학습에 쓰면 안 되는 식별자 |
| Type | 제품 등급 (L/M/H) | 저 50% · 중 30% · 고 20% |
| Air/Process temperature [K] | 공기 온도 / 공정 온도 | 켈빈(K) 단위 |
| Rotational speed [rpm] | 회전수 | |
| Torque [Nm] | 토크(주축에 걸리는 힘) | |
| Tool wear [min] | 공구 사용 누적 시간 | |
| **Machine failure** | **고장 여부 (0/1)** | **오늘 맞혀야 할 답** |
| TWF·HDF·PWF·OSF·RNF | 고장 원인 5종 표시 | 고장이 난 뒤에야 아는 값 — 곧 뺍니다 |

In [ ]:
# 데이터 앞 5행 — 표(DataFrame)의 생김새
df.head()

In [ ]:
# 오늘의 목표 변수: Machine failure — 정상과 고장의 비율은?
cnt = df["Machine failure"].value_counts()
pct = df["Machine failure"].value_counts(normalize=True) * 100
print(f"정상(0): {cnt[0]:,}건 ({pct[0]:.1f}%)")
print(f"고장(1): {cnt[1]:,}건 ({pct[1]:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(["고장 (339건)", "정상 (9,661건)"], [cnt[1], cnt[0]], color=["#C0392B", "#2E7D32"])
for i, v in enumerate([cnt[1], cnt[0]]):
    ax.text(v + 120, i, f"{v:,}건", va="center", fontsize=11)
ax.set_title("데이터의 96.6%가 정상 — 고장은 드뭅니다", fontsize=13)
ax.set_xlim(0, 12000)
plt.tight_layout(); plt.show()

**여기서 잠깐** — 모델이 아무 생각 없이 전부 "정상"이라고만 답해도 정확도는 96.6%입니다.
그래서 오늘은 **정확도가 아니라 "고장을 몇 건 잡았는가"** 를 봅니다.

## STEP 2 — 고장은 어디서 일어나는가

In [ ]:
# 토크(부하) × 공구 마모 — 고장(빨강)이 어디에 모여 있는지 직접 확인
norm = df[df["Machine failure"] == 0]
fail = df[df["Machine failure"] == 1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(norm["Tool wear [min]"], norm["Torque [Nm]"], s=6, alpha=0.25, color="#B0B7C0", label="정상")
ax.scatter(fail["Tool wear [min]"], fail["Torque [Nm]"], s=14, color="#C0392B", label="고장")
ax.axvline(180, ls="--", lw=1, color="#F0641E"); ax.axhline(50, ls="--", lw=1, color="#F0641E")
ax.text(183, ax.get_ylim()[1]*0.95, "마모 180분", color="#F0641E", fontsize=10)
ax.text(2, 51.5, "토크 50Nm", color="#F0641E", fontsize=10)
ax.set_xlabel("공구 마모도 (Tool wear, min)"); ax.set_ylabel("토크 (Torque, Nm)")
ax.set_title("고마모 × 고부하의 교차점에 고장이 군집합니다", fontsize=13)
ax.legend()
plt.tight_layout(); plt.show()

오른쪽 위(마모가 크고 부하도 큰 곳)와 **토크의 위아래 끝**에 빨간 점이 몰립니다. 값 하나에 한계선을 긋는 알람으로는 잡기 어려운 "조합 패턴"입니다.

## STEP 3 — 정답지를 치우고, 공정하게 나눕니다

`TWF·HDF·PWF·OSF·RNF` 는 **고장이 난 뒤에 기록된 원인 표시**입니다. 예측하는 시점에는 알 수 없는 값이므로 입력에서 뺍니다.
넣어 두면 정답지를 보고 시험을 치는 셈이 됩니다. 제안서를 볼 때도 같은 질문을 씁니다 — **"그 값, 예측하는 시점에 알 수 있습니까?"**

In [ ]:
from sklearn.model_selection import train_test_split

# 1) 식별자 제거(과적합 방지) + 사후 플래그 5종 제거(누수 차단)
X = df.drop(columns=["UDI", "Product ID", "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"])
y = df["Machine failure"]

# 2) 문자 등급(Type: L/M/H) → 숫자 표시로 변환(원핫 인코딩)
X = pd.get_dummies(X, columns=["Type"])
X = X[sorted(X.columns)]  # 열 순서 고정 — 누가 실행해도 같은 결과(재현성)

# 3) 층화 분할 — 학습 75% / 평가 25%, 양쪽 고장 비율을 똑같이 유지
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

print(f"학습용: {X_train.shape[0]:,}건 (고장 {int(y_train.sum())}건, {y_train.mean()*100:.2f}%)")
print(f"평가용: {X_test.shape[0]:,}건 (고장 {int(y_test.sum())}건, {y_test.mean()*100:.2f}%)")
print(f"입력 변수 {X.shape[1]}개: {list(X.columns)}")

시험용 2,500건, 그 안의 고장 85건 — 이 85건을 몇 건이나 잡아내는지가 오늘의 승부입니다.

## STEP 4 — 첫 모델: 정확도의 함정을 직접 봅니다
랜덤 포레스트 — 결정 나무 200그루의 다수결입니다. 설비 데이터 같은 표 데이터에서 가장 먼저 써 보는 방법입니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)          # 학습 — 약 5초
pred = model.predict(X_test)         # 평가용 2,500건 판정

acc = accuracy_score(y_test, pred)
rec = recall_score(y_test, pred)
prec = precision_score(y_test, pred)
f1 = f1_score(y_test, pred)
print(f"정확도 {acc*100:.1f}%  |  고장 재현율 {rec*100:.1f}%  |  고장 정밀도 {prec*100:.1f}%  |  F1 {f1:.3f}")

In [ ]:
# 혼동행렬 — 이 표를 읽을 줄 알면 절반은 끝났습니다
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

fig, ax = plt.subplots(figsize=(6.4, 4.6))
mat = [[tn, fp], [fn, tp]]
colors = [["#EAF6EE", "#FFF3CD"], ["#FDEDEB", "#EAF2FF"]]
labels = [[f"TN {tn:,}\n맞게 통과", f"FP {fp}\n헛경보"], [f"FN {fn}\n놓친 고장", f"TP {tp}\n고장을 잡음"]]
for r in range(2):
    for c in range(2):
        ax.add_patch(plt.Rectangle((c, 1 - r), 1, 1, facecolor=colors[r][c], edgecolor="#999"))
        ax.text(c + 0.5, 1.5 - r, labels[r][c], ha="center", va="center", fontsize=13,
                color="#C0392B" if (r, c) == (1, 0) else "#333")
ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(["예측: 정상", "예측: 고장"], fontsize=11)
ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(["실제: 고장", "실제: 정상"], fontsize=11)
ax.set_xlim(0, 2); ax.set_ylim(0, 2)
ax.set_title(f"정확도 {acc*100:.1f}% — 그런데 고장 {tp+fn}건 중 {fn}건을 놓쳤습니다", fontsize=13)
plt.tight_layout(); plt.show()
print(f"→ 놓친 고장(FN) {fn}건 = 전체 고장의 {fn/(tp+fn)*100:.1f}% 미감지")

**정확도는 98%인데, 고장 85건 가운데 40건 안팎을 놓쳤습니다.** 이것이 "정확도의 함정"입니다.

> 여러분 화면의 숫자가 강의자료와 한두 건 다를 수 있습니다(실행 환경에 따른 차이). **숫자는 화면 기준으로 읽습니다.**

---
### ⏸ 2교시는 여기까지입니다. 혼동행렬 그림이 보이면 멈추고 기다려 주세요.
---

## STEP 5 — 처방 두 가지: 고장을 더 무겁게 보게 합니다

In [ ]:
# 처방 1) Class Weight — "고장을 틀리면 벌점을 크게" (코드 한 줄 추가)
model_cw = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
model_cw.fit(X_train, y_train)
pred_cw = model_cw.predict(X_test)
print(f"[Class Weight] 재현율 {recall_score(y_test, pred_cw)*100:.1f}%  |  정밀도 {precision_score(y_test, pred_cw)*100:.1f}%  |  F1 {f1_score(y_test, pred_cw):.3f}")

In [ ]:
# 처방 2) SMOTE — 고장 데이터 사이를 보간해 합성 사례 생성
!pip install -q imbalanced-learn
from imblearn.over_sampling import SMOTE

# 반드시 "학습 데이터에만" 적용합니다 — 시험 데이터는 원본 그대로 둡니다
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"학습 데이터: 고장 {int(y_train.sum()):,}건 → {int(y_train_sm.sum()):,}건 (합성으로 균형 확보)")

model_sm = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_sm.fit(X_train_sm, y_train_sm)
pred_sm = model_sm.predict(X_test)
print(f"[SMOTE] 재현율 {recall_score(y_test, pred_sm)*100:.1f}%  |  정밀도 {precision_score(y_test, pred_sm)*100:.1f}%  |  F1 {f1_score(y_test, pred_sm):.3f}")

In [ ]:
# 세 가지 비교 — 공짜 점심은 없습니다
rows = []
for name, p in [("① 처방 없음", pred), ("② Class Weight", pred_cw), ("③ SMOTE", pred_sm)]:
    t_n, f_p, f_n, t_p = confusion_matrix(y_test, p).ravel()
    rows.append([name, f"{recall_score(y_test,p)*100:.1f}%", f"{precision_score(y_test,p)*100:.1f}%",
                 f"{f1_score(y_test,p):.3f}", f_n, f_p])
comp = pd.DataFrame(rows, columns=["처방", "재현율(고장 잡음)", "정밀도(경보 신뢰)", "F1", "놓친 고장 FN", "헛경보 FP"])
display(comp)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3); w = 0.35
rec_v = [recall_score(y_test, p)*100 for p in [pred, pred_cw, pred_sm]]
pre_v = [precision_score(y_test, p)*100 for p in [pred, pred_cw, pred_sm]]
ax.bar(x - w/2, rec_v, w, label="고장 재현율", color="#0068FF")
ax.bar(x + w/2, pre_v, w, label="고장 정밀도", color="#203864")
for i in range(3):
    ax.text(x[i] - w/2, rec_v[i] + 1, f"{rec_v[i]:.1f}", ha="center", fontsize=10)
    ax.text(x[i] + w/2, pre_v[i] + 1, f"{pre_v[i]:.1f}", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(["처방 없음", "Class Weight", "SMOTE"]); ax.set_ylim(0, 105)
ax.set_title("재현율과 정밀도 — 하나를 올리면 하나가 내려갑니다", fontsize=13); ax.legend()
plt.tight_layout(); plt.show()

**읽는 법** — 표의 "놓친 고장"과 "헛경보" 두 열을 보세요. SMOTE는 놓친 고장을 줄이는 대신 헛경보를 크게 늘립니다.
Class Weight는 이 모델에서는 거의 듣지 않거나 오히려 놓침이 늘기도 합니다(실행 환경에 따라 결과가 조금 다를 수 있습니다).
**기법의 이름이 성능을 보장하지 않습니다.** 그리고 어느 쪽이 나은지는 **그 공장에서 놓침 한 건과 헛경보 한 건이 각각 얼마짜리인가**가 정합니다.

## STEP 6 — 기준선 옮기기: 모델은 그대로, 선만 바꿉니다
모델은 기록마다 "고장일 가능성"을 0~1 사이 숫자로 냅니다. 지금까지는 **0.5를 넘으면 알람**이었습니다. 이 기준선을 내리면 어떻게 될까요?

In [ ]:
# 처방 없는 첫 모델을 그대로 두고, 기준선만 0.5 → 0.1 로 내려 봅니다
prob = model.predict_proba(X_test)[:, 1]          # 고장일 가능성 (0~1)
rows = []
for thr in [0.5, 0.4, 0.3, 0.2, 0.1]:
    t_n, f_p, f_n, t_p = confusion_matrix(y_test, (prob > thr).astype(int)).ravel()
    rows.append([thr, f_n, f_p, f"{t_p / (t_p + f_n) * 100:.1f}%"])
sweep = pd.DataFrame(rows, columns=["기준선", "놓친 고장", "헛경보", "재현율"])
display(sweep)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sweep["기준선"].astype(str), sweep["놓친 고장"], "o-", lw=2.5, color="#EE5A24", label="놓친 고장")
ax.plot(sweep["기준선"].astype(str), sweep["헛경보"], "o-", lw=2.5, color="#1798AE", label="헛경보")
for i, r in sweep.iterrows():
    ax.text(i, r["놓친 고장"] + 4, str(r["놓친 고장"]), ha="center", color="#EE5A24", fontsize=11, fontweight="bold")
    ax.text(i, r["헛경보"] + 4, str(r["헛경보"]), ha="center", color="#1798AE", fontsize=11, fontweight="bold")
ax.set_xlabel("기준선 — 고장일 가능성이 이 값을 넘으면 알람"); ax.set_ylabel("건수 (시험 2,500건 기준)")
ax.set_title("기준선을 내리면 놓침은 줄고 헛경보는 늡니다", fontsize=13); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 직접 돌려 보는 손잡이 — 값을 바꾸고 이 칸을 다시 실행해 보세요
기준선 = 0.3  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
t_n, f_p, f_n, t_p = confusion_matrix(y_test, (prob > 기준선).astype(int)).ravel()
print(f"기준선 {기준선:.2f} → 고장 {t_p + f_n}건 중 {t_p}건 검출 · 놓친 고장 {f_n}건 · 헛경보 {f_p}건")

**읽는 법** — 다시 학습하지 않았는데도 놓친 고장이 크게 줄었습니다. 기준선은 모델의 일부가 아니라 **현장이 정하는 손잡이**입니다.
놓침 한 건이 비쌀수록 기준선을 내리고, 헛경보가 부담일수록 올립니다.

## STEP 7 — 모델은 무엇을 보고 고장이라 했는가 (SHAP)

In [ ]:
# SHAP — 판단 근거를 변수별 기여도로 분해 (설치 약 30초)
!pip install -q shap
import shap

explainer = shap.TreeExplainer(model)              # 베이스라인 모델의 속을 들여다봅니다
sample = X_test.iloc[:500]                          # 속도를 위해 500건 표본
sv = explainer.shap_values(sample)
sv_fail = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)  # "고장" 방향 기여도

shap.summary_plot(sv_fail, sample, show=False, plot_size=(8, 4.5))
plt.title("무엇이 고장 판정을 끌어올리는가 — 변수별 기여도 (붉을수록 값이 큼)", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# 영향이 큰 순서 — 판단에 많이 쓰인 값부터
rank = pd.Series(np.abs(sv_fail).mean(axis=0), index=sample.columns).sort_values(ascending=False)
이름 = {"Rotational speed [rpm]": "회전수", "Torque [Nm]": "토크", "Tool wear [min]": "공구 마모", "Air temperature [K]": "공기 온도", "Process temperature [K]": "공정 온도"}
for i, (k, v) in enumerate(rank.head(5).items(), 1):
    print(f"{i}위  {이름.get(k, k):8s}  평균 기여도 {v:.4f}")

In [ ]:
# 개별 사례 해부 — 모델이 "고장"이라 판정한 한 건, 그 이유는?
import numpy as np
idx = np.where((pred == 1) & (y_test.values == 1))[0][0]   # 실제로 잡아낸 고장 1건
one = X_test.iloc[[idx]]
sv_one = explainer.shap_values(one)
sv_one = sv_one[1] if isinstance(sv_one, list) else (sv_one[:, :, 1] if np.array(sv_one).ndim == 3 else sv_one)

contrib = pd.Series(np.ravel(sv_one), index=one.columns).sort_values(key=abs, ascending=False).head(5)
print("이 설비를 '고장 위험'으로 판정한 근거 TOP 5 (+ 위험 방향 / - 안전 방향):")
for k, v in contrib.items():
    print(f"  {'+' if v >= 0 else '-'}{abs(v):.3f}  {k}  (현재값 {one[k].iloc[0]})")
print("\n→ 이 화면의 쓸모는 하나입니다: 보전 담당자가 보고 \"현장이 아는 것과 맞는다\"고 확인하는 것.")

## 마무리 — 진단 도구 ① 로 넘어갑니다

| 오늘 한 것 | 가져갈 질문 |
|---|---|
| 정확도 98%의 모델을 만들었다 | "고장 몇 건 중 몇 건을 잡았습니까?" |
| 처방을 쓰자 헛경보가 늘었다 | "헛경보는 하루에 몇 번 울립니까?" |
| 기준선만 옮겨도 결과가 바뀌었다 | "기준선은 누가, 무엇을 근거로 정했습니까?" |
| SHAP으로 판단 근거를 보았다 | "현장이 아는 것과 맞습니까?" |

**실습과 현장의 거리** — 오늘은 깨끗한 10,000건과 고장 표시 339건이 미리 준비되어 있었습니다. 현장에서는 그 데이터를 모으는 일이 과제의 대부분입니다.

In [ ]:
# ═══════════ 정리 — 오늘 나온 숫자를 한 번에 봅니다 ═══════════
print("=" * 70)
for name, p in [("처방 없음", pred), ("Class Weight", pred_cw), ("SMOTE", pred_sm)]:
    t_n, f_p, f_n, t_p = confusion_matrix(y_test, p).ravel()
    print(f"[{name:12s}] TN {t_n:4d} | FP {f_p:3d} | FN {f_n:3d} | TP {t_p:3d} | "
          f"정확도 {accuracy_score(y_test, p) * 100:5.1f}% | 재현율 {recall_score(y_test, p) * 100:5.1f}% | 정밀도 {precision_score(y_test, p) * 100:5.1f}%")
print("기준선별 (놓침, 헛경보):", [(float(a), int(b), int(c)) for a, b, c in zip(sweep["기준선"], sweep["놓친 고장"], sweep["헛경보"])])
print("SHAP 순위:", [이름.get(k, k) for k in rank.head(5).index], [round(float(v) * 1000, 1) for v in rank.head(5).values])
print("=" * 70)

---
### 막혔을 때
1. **런타임 연결이 끊겼습니다** → 오른쪽 위 [다시 연결] → 첫 칸부터 차례로 다시 실행합니다 (1~2분)
2. **이름이 정의되지 않았다는 오류(NameError)** → 위의 칸을 건너뛴 것입니다. 첫 칸부터 차례로 실행합니다
3. **숫자가 강의자료와 조금 다릅니다** → 실행 환경에 따라 한두 건 다를 수 있습니다. 화면의 숫자로 읽으시면 됩니다

---
*데이터: AI4I 2020 Predictive Maintenance Dataset, S. Matzka — UCI Machine Learning Repository (CC BY 4.0).*
*ⓒ ㈜에이비에이치 · 제조 AI 심화과정 — 문의: abh@abhcst.com*